# Test: anisotropic flux loss \( \mathbf{q} = -K\nabla T \)

Prototype for a physics-inspired loss on saved inference fields in `../outputs/Our_results_trained_models_2mT.pkl`.

**Definition (per grid point, 2D):**

- \( \nabla T = (\partial_x T, \partial_y T) \)
- \( K = \nabla T \, (\nabla T)^\top \) (rank-1 tensor)
- \( \mathbf{q} = -K \nabla T = -\|\nabla T\|^2 \, \nabla T \)

So **\(\mathbf{q}\) is anti-parallel to \(\nabla T\)** with magnitude \(\|\nabla T\|^3\).

**Loss (scalar):** `MSE(q_pred, q_gt)` with `q_gt` from **COSMO-CLM**.

Model pick/skip lists follow `plot_predictions_by_timeslice.ipynb` and `metric_computation.ipynb` (`TEMP_SKIP_MODELS`).
Vector panels use a **quiver** style similar to wind arrows in `Fig_snapshots.ipynb` / `utils.plotting_utils.show_snapshots`.

**Requires:** PyTorch in the kernel (pickle stores tensors).

In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

try:
    import torch  # needed to unpickle spat_distr tensors
except ImportError as exc:
    raise ImportError(
        'Install torch in this Jupyter kernel to load Our_results_trained_models_2mT.pkl'
    ) from exc

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != 'notebooks':
    NOTEBOOK_DIR = Path('notebooks') if Path('notebooks').exists() else NOTEBOOK_DIR

RESULTS_FILE = (NOTEBOOK_DIR / '../outputs/Our_results_trained_models_2mT.pkl').resolve()
OUTPUT_DIR = (NOTEBOOK_DIR / '../outputs').resolve()
TARGET_VAR = '2mT'
PLOT_VAR = '2mT'
# Ground truth for flux MSE / error maps. Set None if COSMO-CLM is not in the pickle.
# GT_MODEL = 'COSMO-CLM'  # e.g. None on machines without COSMO rows
GT_MODEL = None

def load_results_pickle(path: Path) -> pd.DataFrame:
    """Load inference pickle; fail with a clear message if the file is truncated (common with partial OneDrive sync)."""
    if not path.is_file():
        raise FileNotFoundError(f'Results file not found: {path}')
    size_mb = path.stat().st_size / (1024 ** 2)
    print(f'File size: {size_mb:.1f} MiB')
    try:
        return pd.read_pickle(path)
    except Exception as exc:
        if 'truncated' in str(exc).lower():
            raise RuntimeError(
                f'Pickle at {path} is incomplete or corrupted ({size_mb:.1f} MiB).\n'
                'This is not a COSMO/GT_MODEL issue. Typical fixes:\n'
                '  1) Wait for OneDrive to finish syncing, or copy the file again "always keep on device".\n'
                '  2) Re-run notebooks/models_inference.ipynb to regenerate the pickle.\n'
                '  3) Copy a known-good Our_results_trained_models_2mT.pkl from the cluster/other PC.'
            ) from exc
        raise


print(f'Loading: {RESULTS_FILE}')
results_df = load_results_pickle(RESULTS_FILE)
results_df['time_step'] = pd.to_datetime(results_df['time_step'])

plot_df = results_df[
    (results_df['target_var'] == TARGET_VAR) & (results_df['variable'] == PLOT_VAR)
].copy()

available_models = sorted(plot_df['model'].unique().tolist())
available_times = sorted(plot_df['time_step'].unique())

if GT_MODEL is not None and GT_MODEL not in available_models:
    print(f'Warning: GT_MODEL={GT_MODEL!r} not in pickle — set GT_MODEL = None and add it to SKIP_MODELS.')
    GT_MODEL = None

print(f'Models ({len(available_models)}):', available_models)
print(f'Timestamps ({len(available_times)}):', available_times)
print(f'GT_MODEL for flux MSE: {GT_MODEL}')

Loading: C:\Users\Chuon\OneDrive\Tài liệu\GitHub\Physically-conditioned-latent-diffusion-model-for-temperature\outputs\Our_results_trained_models_2mT.pkl
File size: 922.0 MiB


RuntimeError: Pickle at C:\Users\Chuon\OneDrive\Tài liệu\GitHub\Physically-conditioned-latent-diffusion-model-for-temperature\outputs\Our_results_trained_models_2mT.pkl is incomplete or corrupted (922.0 MiB).
This is not a COSMO/GT_MODEL issue. Typical fixes:
  1) Wait for OneDrive to finish syncing, or copy the file again "always keep on device".
  2) Re-run notebooks/models_inference.ipynb to regenerate the pickle.
  3) Copy a known-good Our_results_trained_models_2mT.pkl from the cluster/other PC.

## Which models to plot vs skip

Edit the three lists below. Defaults mirror `plot_predictions_by_timeslice.ipynb` and drop redundant LMM / interp entries like `metric_computation.ipynb`.

In [ ]:
# --- Reference roles (not compared against themselves) ---
# COSMO-CLM  -> q_gt for MSE
# ERA5       -> coarse driver; usually excluded from ML comparison rows

# Models to SKIP (redundant, weak baselines, or duplicates you do not want on every figure)
SKIP_MODELS = {
    'ERA5',
    'COSMO-CLM',  # temporary: not stored on this machine / not in pickle
    'Linear Interp.',
    'Quadratic Interp.',
    # Redundant LMM aliases (same family as a named ckpt row you keep)
    'LMM_PDE_res',
    'LMM_PDE_res_016_3',
    'LMM_PDE_res_last_3',
    # Add more LMM step variants here if the pickle contains many, e.g.:
    # 'LMM_PDE_res_steps_2', 'LMM_PDE_res_steps_3',
}

# Explicit plot order (None = all models minus SKIP_MODELS, sorted)
model_order = [
    # 'COSMO-CLM',  # enable when GT is available
    'UNET',
    'GAN',
    'LDM_res',
    'LDM_PDE_res',
    'LMM_PDE_res_020_3',
]

# Extra exclusions applied after model_order (same name as plot_predictions_by_timeslice)
exclude_models = ['ERA5']

# --- Decision table (run once; edit lists above, not this table) ---
MODEL_NOTES = {
    'COSMO-CLM': 'PLOT — ground truth T and q_gt',
    'ERA5': 'SKIP — coarse input, not downscaled HR prediction',
    'Linear Interp.': 'SKIP — weak baseline (metric_computation TEMP_SKIP)',
    'Quadratic Interp.': 'SKIP — weak baseline unless you study interp',
    'UNET': 'PLOT — core deterministic baseline',
    'GAN': 'PLOT — core generative baseline',
    'LDM_res': 'PLOT — latent diffusion without PDE training',
    'LDM_PDE_res': 'PLOT — LDM + temperature PDE loss',
    'LMM_PDE_res': 'SKIP — alias; prefer LMM_PDE_res_last_1 or epoch-tagged row',
    'LMM_PDE_res_last_1': 'PLOT — representative last-ckpt one-step LMM',
    'LMM_PDE_res_004_1': 'PLOT — representative mid-training ckpt (example)',
    'LMM_PDE_res_016_3': 'SKIP — duplicate family (metric_computation TEMP_SKIP)',
    'LMM_PDE_res_last_3': 'SKIP — duplicate family unless comparing step count',
    'LMM_PDE_res_steps_1': 'OPTIONAL — same as LMM_PDE_res when multi-ckpt off',
    'LMM_PDE_res_steps_2': 'SKIP — only enable when comparing # MeanFlow steps',
    'LMM_PDE_res_steps_3': 'SKIP — only enable when comparing # MeanFlow steps',
}

decision_rows = []
for m in available_models:
    if m in SKIP_MODELS or m in exclude_models:
        action = 'skip (list)'
    elif model_order is not None and m not in model_order:
        action = 'skip (not in model_order)'
    else:
        action = 'plot'
    note = MODEL_NOTES.get(m, '— add note in MODEL_NOTES if new model name')
    decision_rows.append({'model': m, 'action': action, 'note': note})

decision_df = pd.DataFrame(decision_rows).sort_values(['action', 'model'])
display(decision_df)

## Time slices and plotting options

In [ ]:
time_slices = [
    '2014-04-24 02:00:00',
    '2014-12-28 03:00:00',
    '2016-05-02 04:00:00',
    '2006-05-14 10:00:00',
    '2019-09-02 02:00:00',
]
selected_times = pd.to_datetime(time_slices)

# Grid spacing for gradients (unit cells; same as temperature PDE helpers in-repo)
dx = 1.0
dy = 1.0
grad_eps = 1e-6

# Quiver subsampling (Fig_snapshots uses stride 50 full domain, 10 zoom)
quiver_stride_full = 40
quiver_stride_zoom = 12

# Optional zoom window in pixel indices (row, col) — set None to use full field
zoom_slice = (slice(180, 340), slice(180, 340))

selected_times

In [ ]:
def as_array(value) -> np.ndarray:
    """2-D float field from saved tensor / nested list / numpy."""
    if isinstance(value, (list, tuple)) and len(value) == 1:
        value = value[0]
    if hasattr(value, 'detach'):
        value = value.detach().cpu().numpy()
    elif hasattr(value, 'values'):
        value = value.values
    arr = np.asarray(value, dtype=float).squeeze()
    if arr.ndim != 2:
        raise ValueError(f'Expected 2-D field, got shape {arr.shape}')
    return arr


def resolve_models(df, model_order=None, skip=None, exclude=None):
    available = list(df['model'].drop_duplicates())
    skip = set(skip or [])
    exclude = set(exclude or [])
    if model_order is None:
        models = [m for m in sorted(available) if m not in skip and m not in exclude]
    else:
        models = [m for m in model_order if m not in exclude and m not in skip]
    missing = [m for m in models if m not in set(available)]
    if missing:
        raise ValueError(f'Missing models in pickle: {missing}')
    if not models:
        raise ValueError('No models to plot after filters.')
    return models


def get_temperature_field(df, model: str, ts) -> np.ndarray:
    ts = pd.to_datetime(ts)
    row = df[(df['model'] == model) & (df['time_step'] == ts)]
    if row.empty:
        raise KeyError(f'No row for model={model!r}, time={ts}')
    return as_array(row.iloc[0]['spat_distr'])


def compute_gradients(T: np.ndarray, dx: float = 1.0, dy: float = 1.0) -> tuple[np.ndarray, np.ndarray]:
    """Centered differences with one-sided boundaries (matches metric_computation.ipynb)."""
    T = np.asarray(T, dtype=float)
    H, W = T.shape
    dTdx = np.zeros_like(T)
    dTdy = np.zeros_like(T)
    if W > 2:
        dTdx[:, 1:-1] = (T[:, 2:] - T[:, :-2]) / (2.0 * dx)
    if H > 2:
        dTdy[1:-1, :] = (T[2:, :] - T[:-2, :]) / (2.0 * dy)
    if W > 1:
        dTdx[:, 0] = (T[:, 1] - T[:, 0]) / dx
        dTdx[:, -1] = (T[:, -1] - T[:, -2]) / dx
    if H > 1:
        dTdy[0, :] = (T[1, :] - T[0, :]) / dy
        dTdy[-1, :] = (T[-1, :] - T[-2, :]) / dy
    return dTdx, dTdy


def compute_q_field(T: np.ndarray, dx: float = 1.0, dy: float = 1.0, eps: float = 1e-6):
    """q = -K grad T with K = grad T (grad T)^T  ->  q = -||grad T||^2 grad T."""
    dTdx, dTdy = compute_gradients(T, dx=dx, dy=dy)
    grad_sq = dTdx ** 2 + dTdy ** 2
    scale = grad_sq  # ||grad T||^2
    qx = -scale * dTdx
    qy = -scale * dTdy
    q_mag = np.sqrt(qx ** 2 + qy ** 2 + eps)
    return qx, qy, q_mag, dTdx, dTdy


def flux_mse(q_pred_x, q_pred_y, q_gt_x, q_gt_y) -> float:
    err_x = q_pred_x - q_gt_x
    err_y = q_pred_y - q_gt_y
    return float(np.mean(err_x ** 2 + err_y ** 2))


def validate_times(df, times):
    available_times = set(df['time_step'])
    missing = [ts for ts in times if ts not in available_times]
    if missing:
        raise ValueError(f'Missing timestamps: {missing}')

## Flux MSE table (all selected times × models)

In [ ]:
if GT_MODEL is None:
    print('Skipping flux MSE table (GT_MODEL is None — no reference field for q_gt).')
else:
    validate_times(plot_df, selected_times)
    models_for_metrics = resolve_models(
        plot_df,
        model_order=model_order,
        skip=SKIP_MODELS,
        exclude=exclude_models,
    )
    models_for_metrics = [m for m in models_for_metrics if m != GT_MODEL]

    mse_rows = []
    for ts in selected_times:
        T_gt = get_temperature_field(plot_df, GT_MODEL, ts)
        q_gt_x, q_gt_y, _, _, _ = compute_q_field(T_gt, dx=dx, dy=dy, eps=grad_eps)
        for model in models_for_metrics:
            T_pred = get_temperature_field(plot_df, model, ts)
            qx, qy, _, _, _ = compute_q_field(T_pred, dx=dx, dy=dy, eps=grad_eps)
            mse_rows.append({
                'time_step': ts,
                'model': model,
                'flux_mse': flux_mse(qx, qy, q_gt_x, q_gt_y),
            })

    flux_mse_df = pd.DataFrame(mse_rows)
    pivot = flux_mse_df.pivot(index='model', columns='time_step', values='flux_mse')
    display(pivot)
    display(flux_mse_df.groupby('model')['flux_mse'].mean().sort_values().to_frame('mean_flux_mse'))

## Visualize \(\mathbf{q}\) vectors (temperature background + quiver)

Each panel: **viridis** \(T\), **white quiver** for \(\mathbf{q}\) (subsampled).  
Third column (when enabled): **\(|\mathbf{q}|\)**.  
Rows: models from `model_order` (including COSMO-CLM for reference).

In [ ]:
def _percentile_limits(arr, percentiles=(1.0, 99.0)):
    vmin, vmax = np.nanpercentile(arr, percentiles)
    if vmin == vmax:
        vmax = vmin + 1e-6
    return float(vmin), float(vmax)


def plot_q_panel(
    T,
    qx,
    qy,
    ax_T,
    ax_qmag=None,
    title=None,
    stride=40,
    region_slice=None,
    T_vmin=None,
    T_vmax=None,
    q_key_scale=None,
    cmap_T='viridis',
):
    if region_slice is not None:
        T = T[region_slice]
        qx = qx[region_slice]
        qy = qy[region_slice]

    if T_vmin is None or T_vmax is None:
        T_vmin, T_vmax = _percentile_limits(T)

    im = ax_T.imshow(T, origin='upper', cmap=cmap_T, vmin=T_vmin, vmax=T_vmax)
    H, W = T.shape
    yy, xx = np.mgrid[0:H:stride, 0:W:stride]
    qx_s = qx[::stride, ::stride]
    qy_s = qy[::stride, ::stride]

    # Image row index increases downward; flip qy for visually intuitive arrows
    q_plot = ax_T.quiver(
        xx,
        yy,
        qx_s,
        -qy_s,
        color='w',
        angles='xy',
        scale_units='xy',
        scale=None,
        width=0.003,
        alpha=0.9,
    )
    if q_key_scale is None:
        q_key_scale = np.nanpercentile(np.sqrt(qx ** 2 + qy ** 2), 95)
    if q_key_scale > 0:
        ax_T.quiverkey(
            q_plot,
            0.88,
            0.08,
            q_key_scale,
            f'{q_key_scale:.2e}',
            labelpos='S',
            coordinates='axes',
            color='w',
            fontproperties={'size': 9},
        )
    if title:
        ax_T.set_title(title, fontsize=10)
    ax_T.set_xticks([])
    ax_T.set_yticks([])

    if ax_qmag is not None:
        q_mag = np.sqrt(qx ** 2 + qy ** 2)
        vq0, vq1 = _percentile_limits(q_mag, (2.0, 98.0))
        ax_qmag.imshow(q_mag, origin='upper', cmap='magma', vmin=vq0, vmax=vq1)
        ax_qmag.set_xticks([])
        ax_qmag.set_yticks([])
    return im


def plot_flux_snapshot(
    df,
    ts,
    models,
    gt_model=GT_MODEL,
    stride_full=40,
    stride_zoom=12,
    zoom_slice=None,
    show_q_magnitude=True,
    save_path=None,
):
    ts = pd.to_datetime(ts)
    n_models = len(models)
    n_cols = 3 if show_q_magnitude else 2
    fig, axes = plt.subplots(
        n_models,
        n_cols,
        figsize=(4.2 * n_cols, 3.4 * n_models),
        squeeze=False,
        constrained_layout=True,
    )
    fig.suptitle(f'Anisotropic flux q — {ts}', fontsize=13)

    if gt_model is not None:
        T_ref = get_temperature_field(df, gt_model, ts)
        q_gt_x, q_gt_y, _, _, _ = compute_q_field(T_ref, dx=dx, dy=dy, eps=grad_eps)
    else:
        T_ref = get_temperature_field(df, models[0], ts)
        q_gt_x = q_gt_y = None
    global_T_vmin, global_T_vmax = _percentile_limits(T_ref)
    q_probe = get_temperature_field(df, models[0], ts)
    _qx, _qy, _, _, _ = compute_q_field(q_probe, dx=dx, dy=dy, eps=grad_eps)
    q_key_scale = np.nanpercentile(np.sqrt(_qx ** 2 + _qy ** 2), 95)

    col_titles = ['T + q (full)', 'T + q (zoom)'] + (['|q| (zoom)'] if show_q_magnitude else [])
    for j, t in enumerate(col_titles):
        axes[0, j].set_title(t, fontsize=10)

    for row, model in enumerate(models):
        T = get_temperature_field(df, model, ts)
        qx, qy, _, _, _ = compute_q_field(T, dx=dx, dy=dy, eps=grad_eps)
        if gt_model is not None and model != gt_model:
            mse_val = flux_mse(qx, qy, q_gt_x, q_gt_y)
            label = f'{model}\nMSE={mse_val:.3e}'
        else:
            label = model

        plot_q_panel(
            T,
            qx,
            qy,
            axes[row, 0],
            title=label,
            stride=stride_full,
            region_slice=None,
            T_vmin=global_T_vmin,
            T_vmax=global_T_vmax,
            q_key_scale=q_key_scale,
        )
        ax_mag = axes[row, 2] if show_q_magnitude else None
        plot_q_panel(
            T,
            qx,
            qy,
            axes[row, 1],
            ax_qmag=ax_mag,
            stride=stride_zoom,
            region_slice=zoom_slice,
            T_vmin=global_T_vmin,
            T_vmax=global_T_vmax,
            q_key_scale=q_key_scale,
        )
        axes[row, 0].set_ylabel(model, fontsize=9)

    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
    return fig, axes

In [ ]:
validate_times(plot_df, selected_times)
models_to_plot = resolve_models(
    plot_df,
    model_order=model_order,
    skip=SKIP_MODELS,
    exclude=exclude_models,
)

figures = {}
for ts in selected_times:
    fig, _ = plot_flux_snapshot(
        plot_df,
        ts,
        models=models_to_plot,
        stride_full=quiver_stride_full,
        stride_zoom=quiver_stride_zoom,
        zoom_slice=zoom_slice,
        # save_path=OUTPUT_DIR / f'flux_q_{pd.Timestamp(ts):%Y%m%d_%H%M}.png',
    )
    figures[ts] = fig
    plt.show()

## Optional: \(\mathbf{q}\) error vectors \((\mathbf{q}_{pred} - \mathbf{q}_{gt})\)

Highlights where the anisotropic flux field disagrees with COSMO-CLM.

In [ ]:
def plot_q_error_snapshot(df, ts, models, gt_model=GT_MODEL, stride=20, region_slice=None):
    ts = pd.to_datetime(ts)
    T_gt = get_temperature_field(df, gt_model, ts)
    q_gt_x, q_gt_y, _, _, _ = compute_q_field(T_gt, dx=dx, dy=dy, eps=grad_eps)

    compare_models = [m for m in models if m != gt_model]
    n = len(compare_models)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4), squeeze=False, constrained_layout=True)
    fig.suptitle(f'q error (pred - COSMO) — {ts}')

    for ax, model in zip(axes[0], compare_models):
        T = get_temperature_field(df, model, ts)
        qx, qy, _, _, _ = compute_q_field(T, dx=dx, dy=dy, eps=grad_eps)
        ex = qx - q_gt_x
        ey = qy - q_gt_y
        if region_slice is not None:
            ex = ex[region_slice]
            ey = ey[region_slice]
        err_mag = np.sqrt(ex ** 2 + ey ** 2)
        v0, v1 = _percentile_limits(err_mag, (5, 95))
        ax.imshow(err_mag, origin='upper', cmap='hot', vmin=v0, vmax=v1)
        H, W = err_mag.shape
        yy, xx = np.mgrid[0:H:stride, 0:W:stride]
        ax.quiver(xx, yy, ex[::stride, ::stride], -ey[::stride, ::stride], color='cyan', angles='xy', scale_units='xy', width=0.003)
        ax.set_title(f'{model}\nMSE={flux_mse(qx, qy, q_gt_x, q_gt_y):.3e}')
        ax.set_xticks([])
        ax.set_yticks([])
    return fig


# Example: one timestamp only (requires GT_MODEL)
if GT_MODEL is None:
    print('Skipping q-error panels (GT_MODEL is None).')
elif len(selected_times) > 0:
    fig_err = plot_q_error_snapshot(plot_df, selected_times[0], models_to_plot, stride=quiver_stride_zoom, region_slice=zoom_slice)
    plt.show()

## Torch loss snippet (for training experiments)

Drop-in style matching `TemperatureFieldLosses` gradients; batch shape `[B, 1, H, W]` or `[B, H, W]`.

In [ ]:
import torch
import torch.nn.functional as F


def temperature_gradients_torch(T: torch.Tensor, dx: float = 1.0, dy: float = 1.0):
    if T.dim() == 4:
        T = T[:, 0]
    B, H, W = T.shape
    dTdx = torch.zeros_like(T)
    dTdy = torch.zeros_like(T)
    if W > 2:
        dTdx[:, :, 1:-1] = (T[:, :, 2:] - T[:, :, :-2]) / (2.0 * dx)
    if H > 2:
        dTdy[:, 1:-1, :] = (T[:, 2:, :] - T[:, :-2, :]) / (2.0 * dy)
    if W > 1:
        dTdx[:, :, 0] = (T[:, :, 1] - T[:, :, 0]) / dx
        dTdx[:, :, -1] = (T[:, :, -1] - T[:, :, -2]) / dx
    if H > 1:
        dTdy[:, 0, :] = (T[:, 1, :] - T[:, 0, :]) / dy
        dTdy[:, -1, :] = (T[:, -1, :] - T[:, -2, :]) / dy
    return dTdx, dTdy


def anisotropic_flux_q(T: torch.Tensor, dx: float = 1.0, dy: float = 1.0):
    dTdx, dTdy = temperature_gradients_torch(T, dx=dx, dy=dy)
    grad_sq = dTdx ** 2 + dTdy ** 2
    qx = -grad_sq * dTdx
    qy = -grad_sq * dTdy
    return qx, qy


def anisotropic_flux_mse_loss(T_pred: torch.Tensor, T_gt: torch.Tensor, dx: float = 1.0, dy: float = 1.0):
    qx_p, qy_p = anisotropic_flux_q(T_pred, dx=dx, dy=dy)
    qx_g, qy_g = anisotropic_flux_q(T_gt, dx=dx, dy=dy)
    return F.mse_loss(qx_p, qx_g) + F.mse_loss(qy_p, qy_g)


# Smoke test
_Tp = torch.randn(2, 1, 32, 32)
_Tg = torch.randn(2, 1, 32, 32)
print('example loss:', float(anisotropic_flux_mse_loss(_Tp, _Tg)))